<a href="https://colab.research.google.com/github/MalayThoria/Build_LLM_Guide/blob/main/Shakespeare_LLM_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎭 Build Your Own Shakespeare LLM — From Scratch



---

### What we'll build
A character-level Transformer (the same architecture behind GPT-2, GPT-3, GPT-4) trained on the complete works of Shakespeare. Given a prompt, it will hallucinate convincing Shakespeare-like dialogue.

### What you'll learn
1. **Tokenization** — how text becomes numbers
2. **Embeddings** — how numbers become meaningful vectors
3. **Self-Attention** — the core mechanism that makes Transformers work
4. **Multi-Head Attention** — running attention in parallel
5. **Transformer Blocks** — stacking it all together
6. **Training loop** — how the model actually learns
7. **Text generation** — sampling from the trained model

### Prerequisites
- Basic Python knowledge
- A vague idea of what neural networks are (we'll explain the rest!)
- **GPU enabled in Colab** — go to *Runtime → Change runtime type → T4 GPU*

> 💡 This notebook is inspired by Andrej Karpathy's "Let's build GPT" lecture but rewritten with extra explanations for absolute beginners.

## Step 0 — Setup and Imports

We need just two main libraries: **PyTorch** (the deep learning framework) and Python's built-in `requests` to download the dataset.

PyTorch handles all the heavy math — matrix multiplications, gradients, GPU acceleration. We barely have to think about any of it.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import requests

# Check if we have a GPU available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

if device == 'cpu':
    print("⚠️  No GPU detected! Training will be slow.")
    print("    Go to Runtime → Change runtime type → T4 GPU")
else:
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# Set seed for reproducibility — same random numbers every run
torch.manual_seed(1337)

Using device: cuda
✅ GPU: Tesla T4


## Step 1 — Download Shakespeare

We need text to train on. The **Tiny Shakespeare** dataset is a classic — about 1 MB of all Shakespeare's plays concatenated into one big text file. Small enough to train quickly, large enough to learn real patterns.

This isn't proprietary — it's been used in machine learning tutorials for over a decade.

In [2]:
# Download the dataset
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
response = requests.get(url)
text = response.text

print(f"Length of dataset: {len(text):,} characters")
print(f"\n--- First 500 characters ---\n")
print(text[:500])

Length of dataset: 1,115,394 characters

--- First 500 characters ---

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


**What you just saw:** Raw Shakespeare. Character names in CAPS, dialogue below them. The model will learn this pattern just by reading.

## Step 2 — Tokenization (Text → Numbers)

Computers can't process text directly. We need to convert characters into numbers — this is called **tokenization**.

### Two main approaches:
1. **Character-level** — each character gets its own number ('a'=0, 'b'=1, ...). Small vocabulary (~65 tokens) but long sequences. *We'll use this — simplest to understand.*
2. **Subword/BPE** — what real LLMs use. Common words become 1 token, rare words split into pieces. Much more efficient but more complex.

For our toy model, character-level is perfect. Real production LLMs use libraries like `tiktoken` (OpenAI) or `sentencepiece` (Google).

In [3]:
# Find all unique characters in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(f"Vocabulary ({vocab_size} characters):")
print(''.join(chars))

Vocabulary (65 characters):

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


**Building the encoder/decoder lookup tables:**

We need to go both ways:
- **Encode**: characters → integers (for feeding the model)
- **Decode**: integers → characters (for reading the model's output)

In [4]:
# Create two simple dictionaries (lookup tables)
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}

# Encoder function: string → list of integers
def encode(s):
    return [char_to_int[c] for c in s]

# Decoder function: list of integers → string
def decode(lst):
    return ''.join([int_to_char[i] for i in lst])

# Quick test
print("Encoded 'Hello':", encode("Hello"))
print("Decoded back:   ", decode(encode("Hello")))

Encoded 'Hello': [20, 43, 50, 50, 53]
Decoded back:    Hello


**Now we encode the entire dataset and store it as a PyTorch tensor.**

A tensor is just a multi-dimensional array — like a numpy array but it can live on the GPU and supports automatic gradient computation.

In [5]:
# Encode the full dataset into one giant tensor of integers
data = torch.tensor(encode(text), dtype=torch.long)

print(f"Shape: {data.shape}")
print(f"Data type: {data.dtype}")
print(f"\nFirst 100 tokens: {data[:100]}")

Shape: torch.Size([1115394])
Data type: torch.int64

First 100 tokens: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


## Step 3 — Train/Validation Split

We split the data into:
- **90% training data** — what the model learns from
- **10% validation data** — held out, used to check if the model is overfitting

**Overfitting** = the model memorizes the training data instead of learning general patterns. We catch this by monitoring loss on data the model has never seen.

In [6]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training set:   {len(train_data):,} tokens")
print(f"Validation set: {len(val_data):,} tokens")

Training set:   1,003,854 tokens
Validation set: 111,540 tokens


## Step 4 — Creating Training Batches

We can't feed the model all 1M characters at once. Instead, we sample random chunks.

### Key concepts:
- **`block_size`** (context length) — how many characters the model sees at once. We'll use 256.
- **`batch_size`** — how many sequences we process in parallel. We'll use 64 (more = faster training, more GPU memory).

### Why chunks?
A sequence of 256 characters actually contains **256 separate training examples** — predicting char 2 from char 1, char 3 from chars 1-2, char 4 from chars 1-3, etc.

This is why Transformers are so data-efficient.

In [7]:
# Hyperparameters — feel free to experiment with these!
batch_size = 64        # how many sequences we process in parallel
block_size = 256       # max context length for predictions

def get_batch(split):
    """Generate a small batch of inputs x and targets y."""
    # Pick the right dataset
    data_source = train_data if split == 'train' else val_data

    # Pick `batch_size` random starting positions in the data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))

    # x = the input chunks (shape: batch_size × block_size)
    x = torch.stack([data_source[i:i+block_size] for i in ix])

    # y = the targets (shifted by 1 — what comes next)
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])

    # Move to GPU if available
    x, y = x.to(device), y.to(device)
    return x, y

# Test it out
xb, yb = get_batch('train')
print(f"Input batch shape:  {xb.shape}  (batch_size, block_size)")
print(f"Target batch shape: {yb.shape}")
print(f"\nFirst input row (decoded):  {decode(xb[0].tolist())[:80]!r}")
print(f"First target row (decoded): {decode(yb[0].tolist())[:80]!r}")
print("\n👆 Notice the target is the input shifted by one character.")

Input batch shape:  torch.Size([64, 256])  (batch_size, block_size)
Target batch shape: torch.Size([64, 256])

First input row (decoded):  "\nNot Gloucester's death, nor Hereford's banishment\nNot Gaunt's rebukes, nor Engl"
First target row (decoded): "Not Gloucester's death, nor Hereford's banishment\nNot Gaunt's rebukes, nor Engla"

👆 Notice the target is the input shifted by one character.


## Step 5 — The Key Idea: Self-Attention 🎯

This is **THE** breakthrough that made modern LLMs possible. Take your time here.

### The problem
When predicting the next character, the model needs context from *all previous characters*. But not equally — some characters are more relevant than others.

If we're reading: `"ROMEO: Wherefore art thou ___"`

To predict the next word ("Romeo"), the model needs to:
- Pay **a lot** of attention to "ROMEO:" at the start
- Pay **some** attention to "Wherefore"
- Pay **less** attention to "art" and "thou"

**Self-attention** is the mechanism that learns *how much attention to pay to each previous token*.

### How it works (intuition)

For each token, we compute three vectors:
- **Query (Q)** — "What am I looking for?"
- **Key (K)** — "What do I contain?"
- **Value (V)** — "What information do I offer?"

Then for each token, we compute scores by matching its **Query** against every previous **Key**. High score = high relevance. We use these scores to take a weighted average of all the **Values**.

### Critical: masking
Tokens can only attend to tokens that came **before** them (not future ones — that would be cheating during training). We achieve this with a triangular mask.

Don't worry if this is fuzzy — the code below will make it concrete.

In [8]:
# Let's first see attention in a tiny example before writing the real class
torch.manual_seed(42)

B, T, C = 1, 4, 8  # 1 batch, 4 tokens, 8 features per token
x = torch.randn(B, T, C)
print(f"Input shape: {x.shape}  (Batch, Time/tokens, Channels/features)")

# Single-head attention from scratch
head_size = 8
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)    # (B, T, head_size) — "what I contain"
q = query(x)  # (B, T, head_size) — "what I'm looking for"
v = value(x)  # (B, T, head_size) — "what I offer"

# Attention scores: how much does each query match each key?
# q @ k.T → (B, T, T) — every token's score against every other token
wei = q @ k.transpose(-2, -1) * head_size**-0.5  # scale to keep variance stable
print(f"\nAttention scores shape: {wei.shape}")

# Mask out future tokens (the lower-triangular trick)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
print(f"\nMasked scores (token i can only see tokens 0..i):\n{wei[0]}")

# Softmax → normalize scores to probabilities (they sum to 1 per row)
wei = F.softmax(wei, dim=-1)
print(f"\nAfter softmax (attention weights):\n{wei[0]}")

# Apply weights to values
out = wei @ v  # (B, T, head_size)
print(f"\nOutput shape: {out.shape}")
print("\n💡 Each output position is a weighted blend of all previous positions' values.")

Input shape: torch.Size([1, 4, 8])  (Batch, Time/tokens, Channels/features)

Attention scores shape: torch.Size([1, 4, 4])

Masked scores (token i can only see tokens 0..i):
tensor([[ 0.4428,    -inf,    -inf,    -inf],
        [ 0.0150,  0.0387,    -inf,    -inf],
        [-0.1450, -0.0665, -0.2011,    -inf],
        [ 0.2133,  0.2461, -0.8662,  0.0110]], grad_fn=<SelectBackward0>)

After softmax (attention weights):
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4941, 0.5059, 0.0000, 0.0000],
        [0.3304, 0.3573, 0.3123, 0.0000],
        [0.3135, 0.3239, 0.1065, 0.2561]], grad_fn=<SelectBackward0>)

Output shape: torch.Size([1, 4, 8])

💡 Each output position is a weighted blend of all previous positions' values.


## Step 6 — A Single Attention Head as a Class

Now let's wrap that logic into a clean reusable `nn.Module`. This is one **attention head**.

In [9]:
class Head(nn.Module):
    """One head of self-attention."""

    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        # Three linear projections — no bias (standard in Transformers)
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # The triangular mask (registered as buffer = not a trainable parameter)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        # Dropout: randomly zero out some weights during training → reduces overfitting
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        # Compute K, Q, V from the input
        k = self.key(x)     # (B, T, head_size)
        q = self.query(x)   # (B, T, head_size)

        # Compute attention scores ('affinities')
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5  # scaled dot product
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # causal mask
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # Weighted aggregation of values
        v = self.value(x)
        out = wei @ v
        return out

print("✅ Head class defined.")

✅ Head class defined.


## Step 7 — Multi-Head Attention

One attention head learns one type of relationship. But there are many possible relationships in language — syntax, semantics, long-range vs short-range dependencies.

**Solution:** Run several heads in parallel, each learning something different. Then concatenate their outputs.

Think of it like a committee — each member focuses on a different aspect, then they pool their insights.

In [10]:
class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention running in parallel."""

    def __init__(self, num_heads, n_embd, head_size, block_size, dropout):
        super().__init__()
        # Create `num_heads` independent attention heads
        self.heads = nn.ModuleList([
            Head(n_embd, head_size, block_size, dropout) for _ in range(num_heads)
        ])
        # After concatenating heads, project back to n_embd dimensions
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Run each head and concatenate their outputs along the channel dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        # Project back to original embedding size
        out = self.dropout(self.proj(out))
        return out

print("✅ MultiHeadAttention class defined.")

✅ MultiHeadAttention class defined.


## Step 8 — The Feed-Forward Network

After attention has gathered information from other tokens, each token independently processes that information with a small neural network. This gives the model **per-token computation** — a place to "think" about what attention found.

It's just two linear layers with a ReLU in between. Simple, but essential. The standard pattern is to expand to 4× the embedding dimension internally, then project back.

In [11]:
class FeedForward(nn.Module):
    """A simple two-layer MLP applied to each token independently."""

    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # expand
            nn.ReLU(),                       # non-linearity
            nn.Linear(4 * n_embd, n_embd),  # project back
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

print("✅ FeedForward class defined.")

✅ FeedForward class defined.


## Step 9 — Putting It Together: The Transformer Block

A single Transformer block combines:
1. **Multi-head attention** (communication between tokens)
2. **Feed-forward** (per-token computation)

Plus two crucial tricks:

### Layer Normalization (LayerNorm)
Normalizes activations to have mean 0 and variance 1. Stabilizes training enormously. We use the **pre-norm** variant (norm BEFORE the layer, not after) — this is what modern Transformers do.

### Residual / Skip Connections
We compute `x = x + sublayer(x)` instead of `x = sublayer(x)`. The "+x" creates a **gradient highway** that lets information (and gradients) flow easily through deep networks. Without this, deep Transformers wouldn't train.

These two tricks are what made it possible to stack dozens of layers.

In [12]:
class Block(nn.Module):
    """A Transformer block: attention + feed-forward, with residual connections."""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, n_embd, head_size, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Notice the pattern: x = x + sublayer(layernorm(x))
        x = x + self.sa(self.ln1(x))     # attention with residual
        x = x + self.ffwd(self.ln2(x))   # feed-forward with residual
        return x

print("✅ Transformer Block defined.")

✅ Transformer Block defined.


## Step 10 — The Full GPT Model

Now we assemble everything into a complete model.

### Architecture:
1. **Token embedding** — each character ID becomes a learnable vector
2. **Positional embedding** — encodes WHERE each token is in the sequence (attention itself is position-agnostic)
3. **N Transformer blocks** — stacked, each refines the representation
4. **Final LayerNorm**
5. **Language modeling head** — projects back to vocabulary size to predict next token

### Why positional embeddings?
Attention treats input as a *set*, not a sequence. "Romeo loves Juliet" would look identical to "Juliet loves Romeo" without position info. So we add a learnable position vector to each token's embedding.

In [13]:
# Hyperparameters for the model
n_embd = 384           # embedding dimension (size of each token vector)
n_head = 6             # number of attention heads
n_layer = 6            # number of Transformer blocks stacked
dropout = 0.2          # dropout rate (helps prevent overfitting)

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # Each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # Stack n_layer Transformer blocks
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)        # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)  # output projection

        # Better initialization (from the GPT-2 paper)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Token + position embeddings
        tok_emb = self.token_embedding_table(idx)                          # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T, n_embd)
        x = tok_emb + pos_emb                                              # (B, T, n_embd)

        # Run through the Transformer blocks
        x = self.blocks(x)         # (B, T, n_embd)
        x = self.ln_f(x)           # final layer norm
        logits = self.lm_head(x)   # (B, T, vocab_size) — scores for each next-char prediction

        # If we have targets, compute the loss
        if targets is None:
            loss = None
        else:
            # Reshape for cross_entropy: it expects (N, C) logits and (N,) targets
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        """Generate new tokens by repeatedly predicting the next one."""
        for _ in range(max_new_tokens):
            # Crop context to the last block_size tokens (we can't go beyond our window)
            idx_cond = idx[:, -block_size:]
            # Forward pass to get predictions
            logits, loss = self(idx_cond)
            # Focus only on the last time step (we only care about what comes next)
            logits = logits[:, -1, :]    # (B, C)
            # Convert to probabilities
            probs = F.softmax(logits, dim=-1)
            # Sample from the distribution (not argmax — adds creativity)
            idx_next = torch.multinomial(probs, num_samples=1)
            # Append to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Instantiate and move to GPU
model = GPTLanguageModel().to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Model has {n_params/1e6:.2f}M parameters")

Model has 10.79M parameters


## Step 11 — Helper to Evaluate Loss

Before training, let's write a helper that estimates loss on both train and validation sets. We do this periodically during training to monitor progress and detect overfitting.

`@torch.no_grad()` tells PyTorch we're not computing gradients here — saves memory and speeds things up.

In [14]:
eval_iters = 200  # how many batches to average over

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()  # set model to evaluation mode (disables dropout etc.)
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()  # back to training mode
    return out

print("✅ Eval helper defined.")

✅ Eval helper defined.


## Step 12 — The Training Loop 🚀

This is where the magic happens. The loop is short and standard:

1. **Get a batch** of training data
2. **Forward pass** — compute predictions and loss
3. **Backward pass** — compute gradients (PyTorch does this for us via `loss.backward()`)
4. **Optimizer step** — update weights using the gradients
5. **Zero gradients** — reset for next iteration

We use **AdamW**, an enhanced version of Adam optimizer that handles weight decay properly. It's the standard for training Transformers.

### What loss means
We're using **cross-entropy loss**. A loss of `-ln(1/vocab_size) ≈ 4.17` means the model is guessing randomly. We want to drive this number down — every drop means the model is getting better at predicting Shakespeare.

### Expected training time
- ~5–10 minutes on a Colab T4 GPU
- Final loss should land around 1.4–1.6
- The model has ~10M parameters — tiny by modern standards (GPT-3 is 175B!) but enough to write convincing Shakespeare

In [15]:
# Training hyperparameters
max_iters = 5000       # total training steps
eval_interval = 500    # how often to check val loss
learning_rate = 3e-4   # how big each weight update is (3e-4 is a classic default)

# Optimizer — AdamW is the standard for Transformers
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Starting training... (this should take ~5-10 mins on a T4 GPU)\n")

import time
start_time = time.time()

for iter in range(max_iters):
    # Every eval_interval steps, evaluate on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        elapsed = time.time() - start_time
        print(f"step {iter:5d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f} | elapsed {elapsed:.0f}s")

    # 1. Sample a batch
    xb, yb = get_batch('train')

    # 2. Forward pass — compute predictions and loss
    logits, loss = model(xb, yb)

    # 3. Zero out previous gradients
    optimizer.zero_grad(set_to_none=True)

    # 4. Backward pass — compute gradients
    loss.backward()

    # 5. Update weights
    optimizer.step()

print(f"\n✅ Training complete in {(time.time()-start_time)/60:.1f} minutes!")

Starting training... (this should take ~5-10 mins on a T4 GPU)

step     0 | train loss 4.1885 | val loss 4.1895 | elapsed 59s
step   500 | train loss 1.7786 | val loss 1.9249 | elapsed 388s
step  1000 | train loss 1.4042 | val loss 1.6181 | elapsed 728s
step  1500 | train loss 1.2820 | val loss 1.5329 | elapsed 1066s
step  2000 | train loss 1.1999 | val loss 1.4948 | elapsed 1405s
step  2500 | train loss 1.1366 | val loss 1.4816 | elapsed 1744s
step  3000 | train loss 1.0844 | val loss 1.4914 | elapsed 2082s
step  3500 | train loss 1.0278 | val loss 1.4895 | elapsed 2420s
step  4000 | train loss 0.9783 | val loss 1.5048 | elapsed 2758s
step  4500 | train loss 0.9245 | val loss 1.5314 | elapsed 3096s
step  4999 | train loss 0.8751 | val loss 1.5515 | elapsed 3433s

✅ Training complete in 57.2 minutes!


## Step 13 — Generate Shakespeare! 🎭

Time for the payoff. We start with a single newline character as the "prompt" and let the model generate 1000 new characters one at a time.

Each step it:
1. Looks at the context so far
2. Predicts a probability distribution over the next character
3. **Samples** from that distribution (we don't just pick the most likely — sampling adds creativity)
4. Appends the new character and continues

This is exactly how ChatGPT generates text — one token at a time.

In [16]:
# Start with a single newline character as our seed
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# Generate 1000 new tokens
print("📜 Generated Shakespeare:\n")
print("=" * 60)
generated = model.generate(context, max_new_tokens=1000)
print(decode(generated[0].tolist()))
print("=" * 60)

📜 Generated Shakespeare:


VOLIO:
Go you one thy wife.

ROMEO:
And let the sharpent with thy love rod toward;
For this face I mourn ram on her counsels.
Married and me, Sinto Paul,
That all theirs comes both and all the bosoins:
When I perue, I shall not jest together;
My name hot, this labour's rebellion: let him that we have
remembering the other soldier: and even
Her I am durt toop his palace, O, we know,
So herself and served, fine, who shalt welcomb'd to him.

DUKE:
Must die is conclude, cleanes should what he lives do in honour?

DUKE OF YORK:
What is it?

DUKE OF YORK:
O, speak, full my lord: dear at your cousin!

Fitz Henry, in the garland, as a montory
May kingdomanual cheer'd to me a party orach.
What, are I then?
And I would wish our majesty with our lovince toget?
Sufol Edward, and drown'd time to me can
I speak off me from my son touch.

CATESBY:
Whither, this sun?

CLARENCE:
To joyful wrong is deadly then:
And it worn to lose usear that quake.

KING EDWARD IV:
Uncle, Sir 

## Step 14 — Prompt the Model

You can also feed it a custom prompt and let it continue.

In [19]:
# Custom prompt
prompt = "JULIET: "

# Encode the prompt and put it on the GPU
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

# Generate
output = model.generate(context, max_new_tokens=500)
print(decode(output[0].tolist()))

JULIET: alack, men nay, be gone before.
Why come that dishonour which the general king,
Dares never make suspicion.
For more mine respection grations
With opinious courning it; but I will serve it
To lose it in such gold, or bothers' light,
Who thought my direful kindness to make one.
Look'd opese, the feast will maim gaunt,
And with night of one of my suit
At one elequal for the grainsts
A wingeant. A traitorfa, a print,
And boar supposeer in the minnow of push:
That's the oly Time I bid the county Par


## Step 15 — Experiment! 🧪

Try these to deepen your intuition:

### 1. Tweak hyperparameters
- Increase `n_layer` to 8 → deeper model, more capacity
- Increase `n_embd` to 512 → wider vectors, more expressive
- More `max_iters` → train longer, lower final loss
- *(Watch out for overfitting — train loss << val loss means you've gone too far)*

### 2. Try different sampling strategies
Modify the `generate` method to:
- **Use argmax** (greedy decoding) — `idx_next = probs.argmax(dim=-1, keepdim=True)` — boring, repetitive
- **Use temperature** — divide logits by a temperature value before softmax. Higher temp = more random; lower = more deterministic
- **Use top-k sampling** — only sample from top K most likely tokens

### 3. Train on different text
Replace the dataset URL with any plain-text file — code, Wikipedia, your own writing. The model will mimic its style.

### 4. Save your model
```python
torch.save(model.state_dict(), 'shakespeare_gpt.pth')
# Later, to load:
model.load_state_dict(torch.load('shakespeare_gpt.pth'))
```

## 🎓 What You Just Built

You built a real, working Transformer language model. The architecture you implemented is the same one used by GPT-2, GPT-3, GPT-4, Claude, LLaMA, and every other modern LLM.

### What the production systems add on top:
| Feature | Yours | Real LLMs |
|---------|-------|-----------|
| Parameters | 10M | 70B – 1T+ |
| Training tokens | ~1M | Trillions |
| Training cost | Free (Colab) | $10M – $100M+ |
| Tokenization | Character | Subword (BPE) |
| Fine-tuning (RLHF) | No | Yes |
| Multi-modality | No | Often yes (images, audio) |
| Inference optimizations | None | KV-cache, quantization, flash attention |

But fundamentally — **the architecture is the same**. You now understand how all these systems work under the hood.

### Next steps to go deeper:
- Read the original **"Attention is All You Need"** paper (2017)
- Watch Karpathy's **"Let's build GPT"** lecture on YouTube
- Explore **nanoGPT** — Karpathy's reference implementation
- Learn about **fine-tuning** — adapting pretrained models to specific tasks
- Study **RLHF** (Reinforcement Learning from Human Feedback) — how raw LLMs become helpful assistants

Congratulations on building an LLM from scratch! 🎉